In [ ]:
# NAME : Hassan Naseem 24F-AI-067 (A2)
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import r2_score, accuracy_score, classification_report, confusion_matrix

In [ ]:
df = pd.read_csv("train.csv")

features = [
    "GrLivArea",
    "LotArea",
    "TotRmsAbvGrd",
    "BedroomAbvGr",
    "FullBath",
    "Neighborhood"
]

data = df[features + ["SalePrice"]].copy()

In [ ]:
def remove_outliers(df, cols):
    for col in cols:
        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)
        iqr = q3 - q1
        df = df[df[col].between(q1 - 1.5 * iqr, q3 + 1.5 * iqr)]
    return df


clean = remove_outliers(
    data,
    ["SalePrice", "GrLivArea", "LotArea"]
).reset_index(drop=True)

In [ ]:
clean["Category"] = pd.qcut(
    clean["SalePrice"],
    3,
    labels=["Low", "Medium", "High"]
)

X = pd.get_dummies(
    clean[features],
    columns=["Neighborhood"],
    drop_first=True
)

y_price = clean["SalePrice"]
y_category = clean["Category"].astype(str)

In [ ]:
X_train, X_test, y_price_train, y_price_test, y_cat_train, y_cat_test = train_test_split(
    X,
    y_price,
    y_category,
    test_size=0.2,
    random_state=42,
    stratify=y_category
)

In [ ]:
numeric_cols = [
    "GrLivArea",
    "LotArea",
    "TotRmsAbvGrd",
    "BedroomAbvGr",
    "FullBath"
]

X_train = X_train.copy()
X_test = X_test.copy()

scaler = StandardScaler()

X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_price_train)

price_pred = lr.predict(X_test)

r2 = r2_score(y_price_test, price_pred)

print("Linear Regression R2:", round(r2, 4))

In [ ]:
rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

rf.fit(X_train, y_cat_train)

category_pred = rf.predict(X_test)

accuracy = accuracy_score(y_cat_test, category_pred)

print("Random Forest Accuracy:", round(accuracy, 4))
print(classification_report(y_cat_test, category_pred))

In [ ]:
print("Confusion Matrix:")
print(confusion_matrix(
    y_cat_test,
    category_pred,
    labels=["Low", "Medium", "High"]
))